In [1]:
import numpy as np
import os
import cv2
import glob
from typing import Any

In [2]:
script_path = os.getcwd()
raw_path = os.path.join(script_path, '../34759_final_project_raw')
rect_path = os.path.join(script_path, '../34759_final_project_rect')
left_images_path = os.path.join(raw_path, 'calib/image_02/data/*.png')
right_images_path = os.path.join(raw_path, 'calib/image_03/data/*.png')

left_images = sorted(glob.glob(left_images_path))
right_images = sorted(glob.glob(right_images_path))

chess_board_corners: list[tuple[int,tuple[int, int]]] = [(2,(7,11)), (10,(5,7)), (1,(15,5))]

In [7]:
cv2.destroyAllWindows()

objpoints = [] # 3d point in real world space
imgpoints_left = [] # 2d points in image plane for left camera
imgpoints_right = [] # 2d points in image plane for right camera
board_info = [] # Store info about each board (image index, size, etc.)

In [4]:
def mask_found_region(mask, corners, padding=15):
    """Mask out the region where corners were found"""
    if corners is None or len(corners) == 0:
        return mask

    # Get bounding box of found corners
    x_coords = corners[:, 0, 0]
    y_coords = corners[:, 0, 1]
    x_min = int(np.min(x_coords) - padding)
    y_min = int(np.min(y_coords) - padding)
    x_max = int(np.max(x_coords) + padding)
    y_max = int(np.max(y_coords) + padding)

    # Ensure bounds are within image
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(mask.shape[1], x_max)
    y_max = min(mask.shape[0], y_max)

    # Black out this region in the mask
    mask[y_min:y_max, x_min:x_max] = 0
    return mask

def is_overlapping(corners_new, found_boards, threshold=50):
    """Check if newly found corners overlap with any previously found chessboards"""
    if len(found_boards) == 0:
        return False

    center_new = np.mean(corners_new[:, 0, :], axis=0)

    for _, corners_old in found_boards:
        center_old = np.mean(corners_old[:, 0, :], axis=0)
        distance = np.linalg.norm(center_new - center_old)
        if distance < threshold:
            return True

    return False

In [5]:
def find_chessboard_with_masking(gray_img, chess_sizes: list[tuple[int,tuple[int, int]]], max_attempts=20, visualize=True, img_name=""):
    """Try to find multiple chessboards in an image by masking found ones"""
    mask = np.ones_like(gray_img, dtype=np.uint8) * 255
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE
    flags_sb = cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE + cv2.CALIB_CB_CLUSTERING
    attempt = 0
    consecutive_failures = 0
    found_boards = []
    image_copy = gray_img.copy()
    while attempt < max_attempts and consecutive_failures < 5 and len(found_boards) < 13:
        found_in_this_attempt = False
        masked_img = cv2.bitwise_and(gray_img, gray_img, mask=mask)

        # Try each chessboard size and both orientations
        for number_of_boards, corners_size in chess_sizes:

            # Try both orientations (h,v) and (v,h)
            for orientation in [(corners_size[0], corners_size[1]), (corners_size[1], corners_size[0])]:
                # Try hybrid detection (standard first, then blob if enabled)
                ret_corners, corners = cv2.findChessboardCorners(masked_img, corners_size, flags)
                ret_sb, corners_sb = cv2.findChessboardCornersSB(masked_img, corners_size, flags_sb)

                if ret_sb:
                    corners = corners_sb

                if corners is not None:
                    # Refine corner locations for better accuracy
                    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
                    corners_refined = cv2.cornerSubPix(gray_img, corners, (11,11), (-1,-1), criteria)

                    # Check if this is overlapping with an already found board
                    if not is_overlapping(corners_refined, found_boards, threshold=30):
                        found_boards.append((corners_size, corners_refined))
                        print(f"    Attempt {attempt+1}: Found {orientation[0]}x{orientation[1]}")
                        found_in_this_attempt = True
                        consecutive_failures = 0

                        # Mask out this region
                        mask = mask_found_region(mask, corners_refined, padding=15)

                        # Visualize the mask progression
                        if visualize:
                            mask_vis = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
                            img_with_corners = cv2.cvtColor(image_copy, cv2.COLOR_GRAY2BGR)
                            img_with_corners = cv2.drawChessboardCorners(img_with_corners, orientation, corners_refined, True)
                            overlay = cv2.addWeighted(img_with_corners, 0.7, mask_vis, 0.3, 0)
                            text = f"{img_name} - Board {len(found_boards)}: {orientation[0]}x{orientation[1]}"
                            cv2.putText(overlay, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                            cv2.imshow('Mask Progression', overlay)
                            cv2.waitKey(200)

                        break  # Found one with this variant, move to next attempt

        if not found_in_this_attempt:
            consecutive_failures += 1
            # After 2 consecutive failures with standard method, enable blob detection
            if consecutive_failures == 2:
                print(f"    Switching to SimpleBlobDetector for better center detection...")

        attempt += 1


    print(f"    Final count: {len(found_boards)} chessboards found")
    return found_boards

In [6]:
chessboards_per_image: dict[str, Any]= {}

for idx, (left_img_path, right_img_path) in enumerate(zip(left_images, right_images)):
    print(f"Processing image pair {idx+1}/{len(left_images)}: {os.path.basename(left_img_path)}")

    img_left = cv2.imread(left_img_path)
    img_right = cv2.imread(right_img_path)

    gray_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
    gray_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)

    # Find chessboards in both images with masking to detect multiple boards
    boards_left = find_chessboard_with_masking(gray_left, chess_board_corners, max_attempts=20, visualize=True, img_name=f"Left {idx+1}")
    boards_right = find_chessboard_with_masking(gray_right, chess_board_corners,max_attempts=20, visualize=True, img_name=f"Right {idx+1}")

    print(f"  Found {len(boards_left)} chessboards in left image")
    print(f"  Found {len(boards_right)} chessboards in right image")

    # Match chessboards by size and position between left and right images
    matched_right_indices = set()

    for corners_size_left, corners_left in boards_left:
        # Calculate center of left chessboard
        center_left = np.mean(corners_left[:, 0, :], axis=0)

        # Find matching chessboard in right image (same size, similar y-position)
        best_match = None
        best_match_idx = None
        best_distance = float('inf')

        for idx_right, (corners_size_right, corners_right) in enumerate(boards_right):
            if idx_right in matched_right_indices:
                continue  # Already matched

            # Check if sizes match (comparing the number of corners, not order)
            if sorted(corners_size_left) == sorted(corners_size_right):
                center_right = np.mean(corners_right[:, 0, :], axis=0)
                # For stereo images, y-coordinate should be similar, x can differ
                y_distance = abs(center_left[1] - center_right[1])

                if y_distance < best_distance and y_distance < 150:  # Increased threshold for y-difference
                    best_distance = y_distance
                    best_match = corners_right
                    best_match_idx = idx_right

        if best_match is not None:
            # Create object points for this chessboard size
            objp = np.zeros((corners_size_left[0]*corners_size_left[1], 3), np.float32)
            objp[:,:2] = np.mgrid[0:corners_size_left[0], 0:corners_size_left[1]].T.reshape(-1,2)

            objpoints.append(objp)
            imgpoints_left.append(corners_left)
            imgpoints_right.append(best_match)
            matched_right_indices.add(best_match_idx)
            board_info.append({
                'img_idx': idx,
                'size': corners_size_left,
                'left_path': left_img_path,
                'right_path': right_img_path
            })

            print(f"  Matched {corners_size_left[0]}x{corners_size_left[1]} chessboard (y-diff: {best_distance:.1f}px)")

            # Draw and display the corners
            img_draw_left = img_left.copy()
            img_draw_right = img_right.copy()
            cv2.drawChessboardCorners(img_draw_left, corners_size_left, corners_left, True)
            cv2.drawChessboardCorners(img_draw_right, corners_size_left, best_match, True)

            # Stack images side by side for visualization
            vis = np.hstack([img_draw_left, img_draw_right])
            vis_resized = cv2.resize(vis, (0, 0), fx=0.5, fy=0.5)
            cv2.imshow('Detected Chessboards', vis_resized)
            cv2.waitKey(300)

cv2.destroyAllWindows()
print(f"\nTotal matched chessboard pairs found: {len(objpoints)}")

# Summary of unique chessboard sizes found
unique_sizes = {}
for objp in objpoints:
    size = objp.shape[0]
    # Determine the chessboard dimensions
    for corners in chess_board_corners:
        if corners[0] * corners[1] == size:
            size_str = f"{corners[0]}x{corners[1]}"
            unique_sizes[size_str] = unique_sizes.get(size_str, 0) + 1
            break

print("\nUnique chessboard sizes found:")
for size_str, count in sorted(unique_sizes.items()):
    print(f"  {size_str}: {count} instances")


Processing image pair 1/19: 0000000000.png
    Attempt 1: Found 7x11


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/dragoselul/anaconda3/envs/torch-ex/lib/python3.11/site-packages/cv2/qt/plugins"


    Attempt 1: Found 5x7
    Attempt 1: Found 15x5
    Attempt 2: Found 7x11
    Attempt 2: Found 5x7
    Attempt 3: Found 5x7
    Attempt 4: Found 5x7
    Attempt 5: Found 5x7
    Attempt 6: Found 5x7
    Attempt 7: Found 5x7
    Attempt 8: Found 5x7
    Attempt 9: Found 5x7
    Attempt 10: Found 5x7
    Final count: 13 chessboards found
    Attempt 1: Found 7x11
    Attempt 1: Found 5x7
    Attempt 2: Found 5x7
    Attempt 3: Found 5x7
    Attempt 4: Found 5x7
    Attempt 5: Found 5x7
    Attempt 6: Found 5x7
    Attempt 7: Found 5x7
    Attempt 8: Found 5x7
    Attempt 9: Found 5x7
    Attempt 10: Found 5x7
    Switching to SimpleBlobDetector for better center detection...
    Final count: 11 chessboards found
  Found 13 chessboards in left image
  Found 11 chessboards in right image
  Matched 7x11 chessboard (y-diff: 11.4px)
  Matched 5x7 chessboard (y-diff: 1.9px)
  Matched 5x7 chessboard (y-diff: 12.4px)
  Matched 5x7 chessboard (y-diff: 13.1px)
  Matched 5x7 chessboard (y-diff: 

KeyboardInterrupt: 

In [ ]:
# Visualize all found chessboard pairs
print("\n" + "="*60)
print("VISUALIZING ALL FOUND CHESSBOARD PAIRS")
print("="*60)
print("Press any key to advance to the next pair, ESC to quit.")

for i in range(len(objpoints)):
    info = board_info[i]

    # Load images
    img_left = cv2.imread(info['left_path'])
    img_right = cv2.imread(info['right_path'])

    # Draw corners
    img_draw_left = img_left.copy()
    img_draw_right = img_right.copy()
    cv2.drawChessboardCorners(img_draw_left, info['size'], imgpoints_left[i], True)
    cv2.drawChessboardCorners(img_draw_right, info['size'], imgpoints_right[i], True)

    # Add text overlay
    text = f"Board {i+1}/{len(objpoints)} | Image {info['img_idx']+1} | Size: {info['size'][0]}x{info['size'][1]}"
    cv2.putText(img_draw_left, text, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
    cv2.putText(img_draw_left, "LEFT", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
    cv2.putText(img_draw_right, "RIGHT", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)

    # Stack images side by side
    vis = np.hstack([img_draw_left, img_draw_right])
    vis_resized = cv2.resize(vis, (0, 0), fx=0.35, fy=0.35)
    cv2.imshow('All Found Chessboard Pairs', vis_resized)

    key = cv2.waitKey(0)
    if key == 27:  # ESC key
        break

cv2.destroyAllWindows()
print("\nVisualization complete!")


VISUALIZING ALL FOUND CHESSBOARD PAIRS
Press any key to advance to the next pair, ESC to quit.


In [ ]:
cv2.destroyAllWindows()